## Notes for Sureksha:
- Finished function that processes wells
- Concatenated processed well datasets
- Prepared the input for the models ( arrays ) 


<div style="background-color:#e6f4ea; border-left:5px solid #34a853; padding:10px; border-radius:4px;">
<b>Update:</b> Next we train the models, preferably decision trees models like <code>LigthGBM  </code> 

🛎️ Ping me if still not clear, or if you want to propose a different way or additional step before this ;)
</div>

# 1. Dependencies 

In [ ]:
# import all libraries and functions here ...
import pandas as pd 
import numpy as np 
import os
import glob
import random 
import matplotlib.pyplot as plt
from scipy.signal import correlate 
from sklearn.model_selection import train_test_split 
from tqdm.notebook import tqdm

# 2. Exploratory Data Analysis 

In [ ]:
# constants
#directories
DATA_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
TRAIN_DIR = DATA_DIR + "/train"
TEST_DIR = DATA_DIR + "/test"

#files
HORIZONTAL_FILES = glob.glob(os.path.join(TRAIN_DIR + "/" + "*__horizontal_well.csv"))
TYPEWELL_FILES = glob.glob(os.path.join(TRAIN_DIR + "/" + "*__typewell.csv"))

WINDOW = 25 # for cross correlation 
ROLLING_WINDOW = 30

In [ ]:
# code to see all files in the data/ input directory 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# check horizontal csv file 
def get_horizontal_file():
    file_path = random.choice(HORIZONTAL_FILES)
    df = pd.read_csv(file_path)
    well_ID = os.path.basename(file_path).split("__")[0]
    df['WELL_ID'] = well_ID
    return df, well_ID

In [ ]:
# check typewell csv file 
def get_typewell_file():
    file_path = random.choice(TYPEWELL_FILES)
    df = pd.read_csv(file_path)
    well_ID = os.path.basename(file_path).split("__")[0]
    df['WELL_ID'] = well_ID
    return df 

In [ ]:
# get matching typewell file 
def get_matching_files():
    horizontal_df, well_ID  = get_horizontal_file()
    typewell_file = os.path.join(TRAIN_DIR + "/" + f"{well_ID}__typewell.csv")
    typewell_df = pd.read_csv(typewell_file)
    return horizontal_df, well_ID,typewell_df

In [ ]:
h_df, ID , tw_df = get_matching_files()

In [ ]:
h_df.head()

In [ ]:
tw_df.head()

In [ ]:
h_df.info()

In [ ]:
h_df.describe()

In [ ]:
tw_df.info()

In [ ]:
tw_df['Geology']

In [ ]:
# another well 
h_df_1, _, tw_df_1 = get_matching_files()

In [ ]:
h_df_1.head()

In [ ]:
h_df_1.shape

In [ ]:
h_df_1.info()

In [ ]:
tw_df_1.info()

In [ ]:
def plot_x_y(x, y, x_label , y_label, title):
    plt.figure(figsize = (20,8))
    plt.plot(x, y, linewidth = 0.8)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.grid(True, alpha = 0.3)
    plt.show()

In [ ]:
# MD against TVT 
# illustrates where the horizontal drilling starts (?)
plot_x_y(h_df['TVT'], h_df['MD'], "True Vertical Thickness (TVT)", "Measured Depth (MD)", "MD vs TVT")

In [ ]:
# y against x 
plot_x_y(h_df['MD'], h_df['Z'], "Measured Depth (MD)", "Depth (Z)", "Trajectory")

In [ ]:
plt.figure(figsize=(20, 8))
plt.plot(h_df['MD'], h_df['Z'], linewidth=1.8, label='well trajectory', color='blue')
plt.plot(h_df['MD'], h_df['ANCC'], linewidth=0.8, label='ANCC')
plt.plot(h_df['MD'], h_df['ASTNU'], linewidth=0.8, label='ASTNU')
plt.plot(h_df['MD'], h_df['ASTNL'], linewidth=0.8, label='ASTNL')
plt.plot(h_df['MD'], h_df['EGFDU'], linewidth=0.8, label='EGFDU')
plt.plot(h_df['MD'], h_df['EGFDL'], linewidth=0.8, label='EGFDL')
plt.plot(h_df['MD'], h_df['BUDA'], linewidth=0.8, label='BUDA')
plt.xlabel("MD")
plt.ylabel("Depth (Z)")
plt.title("Well trajectory vs geological formation surfaces")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Quick visual check: do the two GR curves have a similar shape/pattern?
# This does NOT find the actual TVT match — it's just eyeballing texture before
# building the real sliding-window correlation function.

# normalising for a cleaner look 
tw_norm = (tw_df['TVT']- tw_df['TVT'].min())/ (tw_df['TVT'].max() - tw_df['TVT'].min())
h_norm = (h_df['MD'] - h_df['MD'].min()) / (h_df['MD'].max() - h_df['MD'].min())

#plotting 
plt.figure(figsize = (20,8))
plt.plot(tw_norm, tw_df['GR'], linewidth = 0.8 , label = 'Typewell GR')
plt.plot(h_norm, h_df['GR'], linewidth = 0.8, label = 'Horizontal Well GR')
plt.xlabel("X")
plt.ylabel("Gamma Ray (GR)")
plt.title("Normalized GR curve comparison (Typewell vs Horizontal well)")
plt.grid(True, alpha = 0.3)
plt.show()


# 3. Data Proprocessing 

Before we can train anything, we need to turn each well's raw files into one clean table the model can actually learn from. Here's what we're doing for each well:

**Matching horizontal well points to the typewell.**
The horizontal well and typewell are two separate logs — they don't line up row by row, so we can't just merge them directly. Instead, we slide the horizontal well's GR curve across the typewell's GR curve to find where the patterns line up best (cross-correlation). Wherever we find a strong match, we grab the corresponding TVT from the typewell and attach it to that row, along with a score telling us how confident that match is.

**Adding a few engineered features.**
Since the data is sequential (each row depends a bit on the ones around it), we add some rolling-window features — basically local averages/trends over the last few rows — so the model gets a sense of what's happening nearby, not just at a single isolated point.

**Not bothering with missing values.**
There are NaNs scattered through the data, but we're not filling or dropping them — LightGBM handles missing values on its own during training, so it's one less thing to worry about.

**One exception: GR needs to be NaN-free just for the matching step.**
Cross-correlation breaks completely if there's even one NaN in the window, so we make a separate interpolated copy of GR purely to run the matching on. We don't touch the original GR column — that one keeps its real gaps and is what actually goes into the model.

In [ ]:
# function to normalize arrays
def normalize(x):
    return (x - np.mean(x)) / (np.std(x) + 1e-8)

In [ ]:
# dummy example to understand scipy.signal.correlate()
signal = np.array([0,1,0,1,0,1], dtype = int)
reference = np.array([0,1,1,1,1,0,1,0,1,0,1], dtype = int) # where we are looking for similar patterns

# plotting the two before matching
plt.figure(figsize = (20,8))
plt.subplot(1,2,1)
plt.plot(signal)
plt.title("Signal")
plt.grid(True, alpha = 0.3)
plt.subplot(1,2,2)
plt.plot(reference)
plt.title("Reference")
plt.grid(True, alpha = 0.3)
plt.show()

n_signal = normalize(signal)
n_reference = normalize(reference)

corr = correlate(n_reference, n_signal, mode = 'valid')
print("Correlation values: ", corr)
matched_place = np.argmax(corr) # where the signal starts matching the reference 
print("matched_place: ", matched_place)


# x axis 
signal_x = np.arange(matched_place, matched_place + len(signal)) # this is for shifting to start from matched place
reference_x = np.arange(len(reference))

# plotting 
plt.figure(figsize= (20,8))
plt.plot(signal_x, signal, linewidth = 2.5, label = 'Signal')
plt.plot(reference_x, reference, linewidth = 0.8, label = 'Reference')
plt.title("Matching signal and reference")
plt.grid(True, alpha = 0.3)
plt.legend()
plt.show()

### Applying this to the real data.

We'll do the same thing with the actual GR values: use matched_place to find the corresponding index in the typewell, then pull out its TVT and assign it to that row in the horizontal well dataset.


The difference is we won't take the whole array of GR values — we'll slide a small window of GR values at a time, for more precision. Each window produces one matched TVT and one confidence/correlation score, which we'll store in two new columns: matched_tvt and match_score.


Since each window covers several rows but only produces one matched value, we'll assign that value to the center row of the window, rather than to every row it covers — that gives the most representative position for the match. Do not worry about the rest of Matched TVT rows, they will get filled as the window slides :)

In [ ]:
# creating a function to process each well datasets separately. 
def process_well(h_df, tw_df):
    # remove nan from GR columns first for cross correlation 
    h_GR_clean = h_df['GR'].interpolate(limit_direction = 'both')
    tw_GR_clean = tw_df['GR'].interpolate(limit_direction= 'both')
    
    norm_h_GR = normalize(h_GR_clean)
    norm_tw_GR = normalize(tw_GR_clean)

    # initialise the new columns in horizontal file 
    # (We use temporary NumPy arrays here for lightning-fast loop writes)
    matched_TVT_arr = np.full(len(norm_h_GR), np.nan)
    matched_score_arr = np.full(len(norm_h_GR), np.nan)
    tw_TVT_arr = tw_df['TVT'].to_numpy() # Converted to array to avoid .iloc slowdown

    # cross-correlation 
    for i in range(0,len(norm_h_GR)- WINDOW):
        h_seq = norm_h_GR[i:i + WINDOW]
        corr = correlate(norm_tw_GR, h_seq, mode = 'valid') 
        matched_place = np.argmax(corr) 
        confidence_score = corr[matched_place] 
        centre = (i + WINDOW ) // 2


        # assign it to columns
        # (Writing to the NumPy arrays instead of using slower .loc and .iloc)
        matched_TVT_arr[centre] = tw_TVT_arr[(matched_place + WINDOW) // 2]
        matched_score_arr[centre] = confidence_score

    # Assign the fast arrays back to the DataFrame columns once the loop is done
    h_df['matched_TVT'] = matched_TVT_arr
    h_df['matched_score'] = matched_score_arr
        
    # add rolling window features 
    #GR 
    h_df['GR_mean'] = h_df['GR'].rolling(ROLLING_WINDOW, min_periods = 1, center = True).mean()
    h_df['GR_std'] = h_df['GR'].rolling(ROLLING_WINDOW, min_periods = 1, center = True).std()
    # later if we are not sure which rolling window value is the best, we run a loop to add features 
    # with more options (e.g. 10, 20, 100) and let the model decide with feature importance!

    #Z slope 
    h_df['Z_slope'] = h_df['Z'].diff(10) / h_df['MD'].diff(10)
    h_df['Z_slope_mean'] = h_df['Z_slope'].rolling(ROLLING_WINDOW, min_periods = 1, center = True).mean()
    h_df['Z_slope_std'] = h_df['Z_slope'].rolling(ROLLING_WINDOW, min_periods = 1, center = True).std()


    return h_df

In [ ]:
# process and concatenate all wells 
df = []
for h_file in tqdm(HORIZONTAL_FILES):
    well_ID = os.path.basename(h_file).split("__")[0]
    tw_file = os.path.join(TRAIN_DIR + "/" + f"{well_ID}__typewell.csv")
    h_df = pd.read_csv(h_file)
    tw_df = pd.read_csv(tw_file)

    # process the data 
    processed_df = process_well(h_df, tw_df)
    df.append(processed_df)

# concatenate everything 
data = pd.concat(df, ignore_index = True)

In [ ]:
data.head(50)

In [ ]:
data.shape

In [ ]:
# prepare the input to the models 
X = data.drop(columns = ['TVT', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'])
y = data['TVT']

# temp = test + val 
X_train , X_temp,y_train ,  y_temp = train_test_split(X, y, test_size = 0.3, random_state = 42)
X_test , X_val ,  y_test,y_val = train_test_split(X_temp, y_temp, test_size = 0.5, random_state = 42)

X_train = X_train.values 
X_test = X_test.values 
X_val = X_val.values 
y_train = y_train.values 
y_test = y_test.values 
y_val = y_val.values 

print("X_train shape: ", X_train.shape )
print("y_train shape: ", y_train.shape )
print("X_test shape: ", X_test.shape )
print("y_test shape: ", y_test.shape )
print("X_val shape: ", X_val.shape )
print("y_val shape: ",y_val.shape )

# 4. Modeling

# 5. Evaluation